In [ ]:
repository_filter: list[str] = []

In [ ]:
from moderne_visualizations_misc.reusable.data_loader import read_data_table
from moderne_visualizations_misc.reusable import quality_utils as qu
import plotly.express as px
import warnings

warnings.simplefilter("ignore")

# df = read_data_table("../samples/test_quality_issues.csv")
df = read_data_table("../samples/v2/io.moderne.prethink.table.TestQualityIssues.csv")
df = qu.filter_repos(df, repository_filter)
df = qu.add_repo_short(df)

In [ ]:
if len(df) == 0:
    fig = qu.empty_figure()
else:
    language_colors = {
        "java": "#2196F3",
        "javascript": "#FDD835",
        "python": "#4CAF50",
    }

    grouped = (
        df.groupby(["language", "severity", "issueType"])
        .size()
        .reset_index(name="count")
    )

    color_map = {}
    for lang in grouped["language"].unique():
        color_map[lang] = language_colors.get(lang, "#999")
    for _, row in grouped.iterrows():
        color_map[row["severity"]] = color_map.get(row["severity"], language_colors.get(row["language"], "#999"))
        color_map[row["issueType"]] = color_map.get(row["issueType"], language_colors.get(row["language"], "#999"))

    fig = px.sunburst(
        grouped,
        path=["language", "severity", "issueType"],
        values="count",
        color="language",
        color_discrete_map=language_colors,
        title="Test Quality Issues — Language → Severity → Type",
    )
    fig.update_layout(
        height=700,
        margin=dict(l=20, r=20, t=60, b=40),
    )
    fig.update_traces(
        hovertemplate="<b>%{label}</b><br>Count: %{value}<extra></extra>",
    )

fig.show()